In [3]:
# 
import pandas as pd
df = pd.read_csv("Spam.csv", encoding="cp1252")
print(df.head())

     v1                                                 v2 Unnamed: 2  \
0   ham  Go until jurong point, crazy.. Available only ...        NaN   
1   ham                      Ok lar... Joking wif u oni...        NaN   
2  spam  Free entry in 2 a wkly comp to win FA Cup fina...        NaN   
3   ham  U dun say so early hor... U c already then say...        NaN   
4   ham  Nah I don't think he goes to usf, he lives aro...        NaN   

  Unnamed: 3 Unnamed: 4  
0        NaN        NaN  
1        NaN        NaN  
2        NaN        NaN  
3        NaN        NaN  
4        NaN        NaN  


In [5]:
print(df.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5572 entries, 0 to 5571
Data columns (total 5 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   v1          5572 non-null   object
 1   v2          5572 non-null   object
 2   Unnamed: 2  50 non-null     object
 3   Unnamed: 3  12 non-null     object
 4   Unnamed: 4  6 non-null      object
dtypes: object(5)
memory usage: 217.8+ KB
None


In [7]:
y=df.iloc[:,0]
print(y.head())

0     ham
1     ham
2    spam
3     ham
4     ham
Name: v1, dtype: object


In [9]:
x=df.iloc[:,1]
print(x.head())

0    Go until jurong point, crazy.. Available only ...
1                        Ok lar... Joking wif u oni...
2    Free entry in 2 a wkly comp to win FA Cup fina...
3    U dun say so early hor... U c already then say...
4    Nah I don't think he goes to usf, he lives aro...
Name: v2, dtype: object


In [11]:

print(y.value_counts())

v1
ham     4825
spam     747
Name: count, dtype: int64


In [13]:
#Copnvert to lowercase
x=x.str.lower()
print(x[1])

ok lar... joking wif u oni...


In [15]:
#Remove Punctutaions
import string
exclude=string.punctuation
def remove_punc(text):
    return text.translate(str.maketrans('','',exclude))

x=x.apply(remove_punc)
print(x[7])

as per your request melle melle oru minnaminunginte nurungu vettam has been set as your callertune for all callers press 9 to copy your friends callertune


In [17]:
#Toeknise words
from nltk.tokenize import word_tokenize
def tokeniser(text):
    return word_tokenize(text)

x=x.apply(tokeniser)


In [19]:
print(x[0])

['go', 'until', 'jurong', 'point', 'crazy', 'available', 'only', 'in', 'bugis', 'n', 'great', 'world', 'la', 'e', 'buffet', 'cine', 'there', 'got', 'amore', 'wat']


In [25]:
class mydataset(Dataset):
    def __init__(self,x,y):
        self.x=x
        self.y=y
        self.n_samples=len(self.x)

    def __len__(self):
        return self.n_samples

    def __getitem__(self,index):
        return self.x.iloc[index],self.y.iloc[index]

In [27]:
from torch.utils.data import random_split
dataset=mydataset(x,y)
train_size=int(0.8*len(dataset))
test_size=len(dataset)-train_size
train_dataset,test_dataset=random_split(dataset, [train_size,test_size])

In [29]:
x_train=train_dataset[:][0]
y_train=train_dataset[:][1]

In [31]:
x_test=test_dataset[:][0]
y_test=test_dataset[:][1]

In [33]:
#Make a vocablary
vocab={
    '<PAD>':0,
    '<UNK>':1
}

In [35]:
for sentence in x_train:
    for word in sentence:
        if word not in vocab:
            vocab[word]=len(vocab)

print(len(vocab))

8531


In [37]:
#Assign ID to each word in vocbluary for x_train
def sentence2int(text):
    encoded=[]
    for word in text:
        if word in vocab:
            encoded.append(vocab[word])
        else:
            encoded.append(1)
    return encoded

x_train=x_train.apply(sentence2int)
x_train.head()

4315    [2, 3, 4, 5, 6, 7, 8, 2, 9, 10, 11, 12, 11, 13...
2887    [2, 19, 20, 21, 22, 7, 23, 24, 25, 26, 9, 21, ...
2759    [9, 32, 33, 34, 35, 36, 37, 38, 2, 39, 40, 2, ...
1734                      [2, 47, 48, 49, 42, 50, 31, 51]
755                                  [16, 52, 22, 11, 53]
Name: v2, dtype: object

In [39]:
x_test=x_test.apply(sentence2int)

In [41]:
print(type(x_test.iloc[0]))
print(x_test.iloc[0])

<class 'list'>
[1, 59, 170, 16, 3432, 634]


In [23]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
import torch.nn.functional as F
import torch.optim as optim

D:\ANACONDA\Lib\site-packages\torch\cuda\__init__.py:61: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]


In [43]:
print(len(vocab))

8531


In [45]:
class mymodel(nn.Module):
    def __init__(self):
        super().__init__()
        self.embedding=nn.Embedding(len(vocab),100)
        self.lstm=nn.LSTM(input_size=100,hidden_size=128,batch_first=True)
        self.dropout=nn.Dropout(0.5)
        self.fc1=nn.Linear(128,2)

    def forward(self,x):
        x=self.embedding(x)
        output, (h_n, c_n)=self.lstm(x)
        final_state=self.dropout(h_n[-1])
        final_state=self.fc1(final_state)
        return final_state       

In [47]:
# To find maxlength sentence in each batch and add padding accordingly
def collate(batch):
    maxi=0
    x=[]
    y=[]
    for sentence,labels in batch:
        maxi=max(maxi,len(sentence))
        x.append(sentence)
        y.append(labels)

    for i in range(len(x)):
        x[i]=x[i] + [0] * (maxi - len(x[i]))

    x=torch.tensor(x,dtype=torch.long)
    y=torch.tensor(y,dtype=torch.long)

    return x,y

In [49]:
#Handle class imabalance by giving more weight to spam so that every spam misclassification is penalised more
ham=y.value_counts()["ham"]
spam=y.value_counts()["spam"]
total=ham+spam
weight_ham=total/(2*ham)
weight_spam=total/(2*spam)
weights=torch.tensor([weight_ham,weight_spam],dtype=torch.float)

In [51]:
#Shift everything to GPU
device=torch.device(
    "cuda" if torch.cuda.is_available()
    else "cpu"
)

weights = weights.to(device)

In [53]:
label_map = {
    "ham": 0,
    "spam": 1
}

y_train = y_train.map(label_map)
y_test = y_test.map(label_map)

In [55]:
print(y_train.head())

4315    0
2887    0
2759    0
1734    0
755     0
Name: v1, dtype: int64


In [57]:
train_dataset=mydataset(x_train,y_train)
test_dataset=mydataset(x_test,y_test)


train_loader=DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True,
    collate_fn=collate
)

test_loader=DataLoader(
    test_dataset,
    batch_size=32,
    shuffle=False,
    collate_fn=collate
)

In [59]:
model=mymodel()
model=model.to(device)
criterion=nn.CrossEntropyLoss(weight=weights)
optimiser=optim.Adam(model.parameters(),lr=0.001)

In [60]:
model.train()
for epoch in range(75):
    epoch_loss=0
    for x_train,y_train in train_loader:
        x_train = x_train.to(device)
        y_train = y_train.to(device)
        optimiser.zero_grad()
        output=model(x_train)
        loss=criterion(output,y_train)
        loss.backward()
        optimiser.step()
        epoch_loss=epoch_loss+loss.item()
        
    print("Loss for Epoch ",epoch,":",epoch_loss)


Loss for Epoch  0 : 96.71633297204971
Loss for Epoch  1 : 46.08968970924616
Loss for Epoch  2 : 28.879849327728152
Loss for Epoch  3 : 18.121299535036087
Loss for Epoch  4 : 15.86428511608392
Loss for Epoch  5 : 10.105181724764407
Loss for Epoch  6 : 5.983768446370959
Loss for Epoch  7 : 4.0442698682891205
Loss for Epoch  8 : 2.759804291301407
Loss for Epoch  9 : 4.083552588708699
Loss for Epoch  10 : 2.8007784750079736
Loss for Epoch  11 : 1.3337200202513486
Loss for Epoch  12 : 0.4780155310872942
Loss for Epoch  13 : 0.9362880699918605
Loss for Epoch  14 : 0.3971574268362019
Loss for Epoch  15 : 0.29989742090401705
Loss for Epoch  16 : 4.334939832464443
Loss for Epoch  17 : 1.3439869675203227
Loss for Epoch  18 : 0.35430293300305493
Loss for Epoch  19 : 0.14003571272769477
Loss for Epoch  20 : 0.1369474115272169
Loss for Epoch  21 : 0.10377737835733569
Loss for Epoch  22 : 0.042413799339556135
Loss for Epoch  23 : 0.03233058396290289
Loss for Epoch  24 : 0.02716436254195287
Loss for 

In [63]:
correct=0
total=0

model.eval()
with torch.no_grad():
    for x_test,y_test in test_loader:
        x_test = x_test.to(device)
        y_test = y_test.to(device)
        outputs=model(x_test)
        _, predicted = torch.max(outputs,1)
        total += y_test.size(0)
        correct += (predicted == y_test).sum().item()

accuracy = 100 * correct / total

print(f"Accuracy: {accuracy:.2f}%")

Accuracy: 97.67%


In [65]:

test_messages = [
    # Obvious spam
    "Congratulations you have won a free prize worth 1000 dollars call now to claim",
    "URGENT your account has been selected for a cash reward click here to collect",
    "You have been chosen to receive a free iPhone text WIN to 80808",
    
    # Obvious ham
    "Hey are you coming to the party tonight",
    "Can you pick up some milk on the way home",
    "I will be late for dinner today sorry",
    
    # Tricky ones
    "Your order has been shipped and will arrive by tomorrow",
    "Please call me when you get this message its important",
    "Free workshop on machine learning this saturday at the university",
]

labels = [
    "spam", "spam", "spam",
    "ham", "ham", "ham",
    "ham", "ham", "ham",
]

exclude = string.punctuation

model.eval()
with torch.no_grad():
    for msg, true_label in zip(test_messages, labels):
        # same preprocessing as training
        processed = msg.lower()
        processed = processed.translate(str.maketrans('', '', exclude))
        tokens = word_tokenize(processed)
        encoded = [vocab.get(word, 1) for word in tokens]  # 1 = <UNK>
        
        input_tensor = torch.tensor([encoded], dtype=torch.long).to(device)
        output = model(input_tensor)
        _, predicted = torch.max(output, 1)
        pred_label = "spam" if predicted.item() == 1 else "ham"
        
        status = "✅" if pred_label == true_label else "❌"
        print(f"{status} True: {true_label:4s} | Predicted: {pred_label:4s} | {msg}")

✅ True: spam | Predicted: spam | Congratulations you have won a free prize worth 1000 dollars call now to claim
✅ True: spam | Predicted: spam | URGENT your account has been selected for a cash reward click here to collect
✅ True: spam | Predicted: spam | You have been chosen to receive a free iPhone text WIN to 80808
✅ True: ham  | Predicted: ham  | Hey are you coming to the party tonight
✅ True: ham  | Predicted: ham  | Can you pick up some milk on the way home
✅ True: ham  | Predicted: ham  | I will be late for dinner today sorry
❌ True: ham  | Predicted: spam | Your order has been shipped and will arrive by tomorrow
✅ True: ham  | Predicted: ham  | Please call me when you get this message its important
✅ True: ham  | Predicted: ham  | Free workshop on machine learning this saturday at the university


In [67]:
torch.save(model.state_dict(), "spam_classifier.pth")

In [69]:
import pickle

with open("vocab.pkl", "wb") as f:
    pickle.dump(vocab, f)